### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
### Groq API and usage with langchain

import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv('OPENAI_API_KEY')

groq_api_key = os.getenv("GROQ_API_KEY")

In [7]:
from langchain_groq import ChatGroq
model = ChatGroq(model ="openai/gpt-oss-120b", groq_api_key=groq_api_key )
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x79b321d74a40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x79b321d746e0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello! How are you?")
]

response = model.invoke(messages)
response

AIMessage(content='Bonjour\u202f! Comment ça va\u202f?', additional_kwargs={'reasoning_content': 'We need to translate English to French. The user says "Hello! How are you?" We respond with French translation: "Bonjour ! Comment ça va ?" Possibly "Bonjour ! Comment vas-tu ?" Choose a polite form. Use "Bonjour! Comment ça va?" Probably correct.'}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 87, 'total_tokens': 160, 'completion_time': 0.15720932, 'completion_tokens_details': {'reasoning_tokens': 56}, 'prompt_time': 0.003444701, 'prompt_tokens_details': None, 'queue_time': 0.044637729, 'total_time': 0.160654021}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ccc9d-5497-7c40-ba4c-3ebbc0972c79-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 73, 'total_tokens': 160, 'outp

In [9]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(response)

'Bonjour\u202f! Comment ça va\u202f?'

Using LCEL - we can chain the components 
* This means the output of one componnent will automatically taken as the input of the next component and then we get the final output
* We do not have to explicitly get the output1 and give it as input to next manually.
* The above was manual, now this is with chain, we can see both the outputs match , hence LCEL is working fine


In [ ]:
chain = model|parser 
chain.invoke(messages)

'Bonjour\u202f! Comment ça va\u202f?'

Prompt templates

* Here we will see how to create chain with prompt templates
* Here prompt template lets us convert the prompt also in to langchain object, so that we can include that also in the chain and automate the whole thing.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

generic_mannual_template = "Translate the following into {language}:"

prompt = ChatPromptTemplate.from_messages(
    [("system",generic_mannual_template),("user","{text}")]
)




Here language and text are placeholder words, like parameters which can be passed as a dictionary to the prompt.invoke() method


In [14]:
#Lets see how this looks like  and functions
# NOTE: We can invoke individual components as well as we can invoke a chain too

result = prompt.invoke({"language":"French", "text":"Hello"})
result
#This outputs a ChatPromptValue object

ChatPromptValue(messages=[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})])

In [18]:
# to view it in more detail
result.to_messages()
# Here we can see clearly what is what and all

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

Now, we will create a chain by including the prompt also 


In [20]:
chain = prompt|model|parser
chain.invoke({"language":"French", "text":"Hello"})

'Bonjour'